# Phase 2b (Exploratory) — SNV/CNV Split Matrices, CNV Driver Filter, Batch-Effect Probe

**Status: exploratory prototype, not yet folded into the main pipeline.** This
notebook investigates three open methodology questions raised after Phase 2:

1. Should the CNV side of `alterations_long.parquet` be filtered down to
   likely **driver** events before feeding a CNV-specific co-occurrence
   analysis, instead of treating every deep amp/deep del as equally
   meaningful?
2. Is it worth splitting Phase 2's single blended matrix into three
   (**SNV-only**, **CNV-only**, **Combined**), and what does that actually
   buy us?
3. Does a panel/batch effect (167 different sequencing panels) meaningfully
   distort the results, and does a stratified (panel-level) check catch it?

Run on a **3-cancer-type pilot** (Breast Cancer, Non-Small Cell Lung Cancer,
Bladder Cancer -- chosen for known, well-characterized CNV biology to sanity
check against) rather than the full 69 cancer types, to get a fast, grounded
answer before deciding whether to commit to a full rebuild.

**Headline result, found while building the batch-effect probe (Section 5):
this surfaced a real, previously undetected data problem** -- 21 of the 48
panels flagged `cna_capable` in Phase 1 report **zero** deep CNA calls
across their entire sample population, despite their own metadata claiming
CNA capability. This affects ~21.6% of all "CNA-capable" samples and
disproportionately hits some cancer types (Glioma 35.6%, NSCLC 24.1%). See
Section 5 for the full evidence and Section 6 for what to do about it.


In [1]:
import itertools
import json
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import fisher_exact
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
pd.set_option("future.no_silent_downcasting", True)

DATA_DIR = Path.cwd().parent / "data"
EXTERNAL_DIR = DATA_DIR / "external"
PROCESSED_DIR = DATA_DIR / "processed"

alterations = pd.read_parquet(PROCESSED_DIR / "alterations_long.parquet")
panel_coverage = pd.read_parquet(PROCESSED_DIR / "panel_gene_coverage.parquet")
clinical = pd.read_parquet(PROCESSED_DIR / "clinical_tidy.parquet")
consensus_per_ct = pd.read_parquet(PROCESSED_DIR / "consensus_genes_per_cancer_type.parquet")
genes_by_ct = consensus_per_ct.groupby("CANCER_TYPE")["Hugo_Symbol"].apply(set).to_dict()

with open(EXTERNAL_DIR / "oncokb_curated_genes.json") as f:
    oncokb_genes = json.load(f)

print(alterations.shape, panel_coverage.shape, clinical.shape)


(1974322, 13) (53291, 4) (271837, 19)


## 1. CNV driver-consistency annotation

Not every deep amplification/deletion is a driver event -- some ride along
on a large chromosomal-arm-level CNA that happens to sweep up nearby genes
too (passengers). OncoKB's **gene-level role** curation (already used for
the Phase 1 two-hit TSG feature: 406 TSG genes) also curates 463
**ONCOGENE** and 70 **ONCOGENE_AND_TSG** genes -- enough to build a simple,
free, tokenless driver-direction check, generalized from the two-hit
feature to the whole CNV set (not just TSG-paired-with-mutation):

- **TSG** + `Deep_Deletion` -> consistent (loss-of-function fits a
  tumour-suppressor)
- **ONCOGENE** + `Amplification` -> consistent (gain-of-function fits an
  oncogene)
- **ONCOGENE_AND_TSG** -> either direction consistent
- Everything else (not OncoKB-curated, or curated `INSUFFICIENT_EVIDENCE` /
  `NEITHER`) -> **Unannotated**, not silently dropped -- we can't verify
  direction for these, which is different from verifying they're *wrong*.
- A TSG amplified or an oncogene deleted -> **Non_Consistent** -- doesn't
  match the expected mechanism, more likely a passenger.


In [2]:
GENE_ROLE = {g["hugoSymbol"]: g["geneType"] for g in oncokb_genes}
CONSISTENT_DIRECTION = {
    "TSG": {"Deep_Deletion"},
    "ONCOGENE": {"Amplification"},
    "ONCOGENE_AND_TSG": {"Deep_Deletion", "Amplification"},
}

cnv = alterations[alterations["Alteration_Type"] == "CNV"].copy()
cnv["Gene_Role"] = cnv["Hugo_Symbol"].map(GENE_ROLE)


def _driver_status(row):
    role = row["Gene_Role"]
    if role not in CONSISTENT_DIRECTION:
        return "Unannotated"
    return "Driver_Consistent" if row["CNA_Direction"] in CONSISTENT_DIRECTION[role] else "Non_Consistent"


cnv["Driver_Status"] = cnv.apply(_driver_status, axis=1)
print(cnv["Driver_Status"].value_counts())
print((cnv["Driver_Status"].value_counts(normalize=True) * 100).round(1).astype(str) + "%")


Driver_Status
Driver_Consistent    236755
Non_Consistent        85254
Unannotated           55207
Name: count, dtype: int64
Driver_Status
Driver_Consistent    62.8%
Non_Consistent       22.6%
Unannotated          14.6%
Name: proportion, dtype: object


In [3]:
print("Top 15 Non_Consistent genes (candidate passenger CNVs):")
print(cnv.loc[cnv["Driver_Status"] == "Non_Consistent", "Hugo_Symbol"].value_counts().head(15))

print("\nTop 15 Unannotated genes (not OncoKB TSG/ONCOGENE):")
print(cnv.loc[cnv["Driver_Status"] == "Unannotated", "Hugo_Symbol"].value_counts().head(15))


Top 15 Non_Consistent genes (candidate passenger CNVs):
Hugo_Symbol
RECQL4     3171
RAD21      3000
CDK12      2678
SDHA       2231
CEBPA      1819
NBN        1745
ASXL1      1590
RTEL1      1416
NFKBIA     1341
SDHC       1293
BRIP1      1285
CRLF2      1008
SOX17       926
KMT2B       881
PRKAR1A     848
Name: count, dtype: int64

Top 15 Unannotated genes (not OncoKB TSG/ONCOGENE):
Hugo_Symbol
WHSC1L1     3154
AGO2        2147
BCL2L1      1971
RAD52       1495
RARA        1272
TCEB1       1172
HIST2H3C    1072
TEK         1070
PREX2        999
PRDM14       985
IKZF1        739
KAT6A        715
RAD54B       711
H3F3C        664
PIK3C2G      641
Name: count, dtype: int64


**Reading this**: 62.8% of deep CNA calls are Driver_Consistent, 22.6%
Non_Consistent, 14.6% Unannotated. The Non_Consistent list is plausible --
`RAD21`, `CDK12`, `SDHA`, `NBN`, `ASXL1`, `BRIP1` etc. are mostly DNA-repair
/ chromatin genes that sit near known amplicons or fragile regions and can
get swept up passively. This is a real, usable filter, not a coin flip.

**Open decision for Jason**: use `Driver_Consistent` only (strict) for a
CNV-only analysis, or `Driver_Consistent + Unannotated` (loose, since
"unannotated" isn't "wrong")? Section 3 onward uses the strict definition.


## 2. CNA-capable-aware coverage (the piece this unlocks)

Phase 1 built a `cna_capable` flag (only 48 of 167 panels can call CNAs at
all) but explicitly did **not** use it to gate the gene list, to avoid
sacrificing power for the ~81% mutation-driven majority in the combined
analysis. A **dedicated CNV-only matrix** removes that tradeoff entirely --
restricting its coverage check to CNA-capable panels only costs nothing
elsewhere. Rebuilding the per-cancer-type coverage check (>=80% of the
*CNA-capable-eligible* population, >=100 tested) for the 3 pilot cancer
types:


In [4]:
PILOT_CANCER_TYPES = ["Breast Cancer", "Non-Small Cell Lung Cancer", "Bladder Cancer"]

cna_capable_panels = panel_coverage[panel_coverage["cna_capable"]]
ct_samples_all = clinical[clinical["CANCER_TYPE"].isin(PILOT_CANCER_TYPES)][["SAMPLE_ID", "CANCER_TYPE", "SEQ_ASSAY_ID"]]


def per_ct_gene_coverage(coverage_source, ct_samples, cutoff=0.80, min_tested=100):
    # Denominator = samples actually eligible under coverage_source (e.g. on
    # a CNA-capable panel), NOT the cancer type's full population -- else
    # samples that could never produce ANY CNV call drag every gene's
    # coverage fraction down for no reason.
    eligible_samples = ct_samples[ct_samples["SEQ_ASSAY_ID"].isin(coverage_source["SEQ_ASSAY_ID"])]
    merged = coverage_source.merge(eligible_samples, on="SEQ_ASSAY_ID", how="inner")
    n_ct = eligible_samples.groupby("CANCER_TYPE")["SAMPLE_ID"].nunique()
    g = merged.groupby(["CANCER_TYPE", "Hugo_Symbol"])["SAMPLE_ID"].nunique().reset_index()
    g = g.merge(n_ct.rename("n_ct_samples"), left_on="CANCER_TYPE", right_index=True)
    g["coverage_frac"] = g["SAMPLE_ID"] / g["n_ct_samples"]
    qualified = g[(g["coverage_frac"] >= cutoff) & (g["SAMPLE_ID"] >= min_tested)]
    return qualified.groupby("CANCER_TYPE")["Hugo_Symbol"].apply(set).to_dict()


cnv_genes_by_ct = per_ct_gene_coverage(cna_capable_panels, ct_samples_all)
for ct in PILOT_CANCER_TYPES:
    n_cna = ct_samples_all[(ct_samples_all["CANCER_TYPE"] == ct) & (ct_samples_all["SEQ_ASSAY_ID"].isin(cna_capable_panels["SEQ_ASSAY_ID"]))]["SAMPLE_ID"].nunique()
    print(f"{ct:28s} | CNA-capable samples: {n_cna:6d} | CNA-capable-covered genes: {len(cnv_genes_by_ct.get(ct, set())):4d} | main (all-panel) list: {len(genes_by_ct.get(ct, set())):4d}")


Breast Cancer                | CNA-capable samples:  19390 | CNA-capable-covered genes:  223 | main (all-panel) list:  167
Non-Small Cell Lung Cancer   | CNA-capable samples:  32199 | CNA-capable-covered genes:  246 | main (all-panel) list:   83
Bladder Cancer               | CNA-capable samples:   6517 | CNA-capable-covered genes:  314 | main (all-panel) list:  267


## 3. Generalized matrix builder / Fisher test

Same logic as `02_phase2_comutation_matrix.ipynb` Sections 3-5, generalized
to take the alteration subset (SNV-only / CNV-driver-only / Combined) and
coverage subset (all panels / CNA-capable only) as parameters, so the exact
same tested-machinery runs all three matrices.


In [5]:
MIN_GENE_ALT_FRACTION = 0.03
MIN_GENE_ALT_ABS = 5


def get_qualifying_genes_g(sub_alt, n_samples, allowed_genes):
    gene_sample_counts = sub_alt.groupby("Hugo_Symbol")["Sample_ID"].nunique()
    min_count = max(MIN_GENE_ALT_ABS, MIN_GENE_ALT_FRACTION * n_samples)
    freq_qualified = set(gene_sample_counts[gene_sample_counts >= min_count].index)
    return sorted(freq_qualified & allowed_genes)


def build_matrices_g(cancer_type, allowed_genes, alt_source, coverage_source):
    ct_samples = clinical.loc[clinical["CANCER_TYPE"] == cancer_type, ["SAMPLE_ID", "SEQ_ASSAY_ID"]]
    ct_samples = ct_samples.drop_duplicates(subset="SAMPLE_ID").set_index("SAMPLE_ID")
    n_samples = len(ct_samples)

    sub_alt = alt_source.loc[alt_source["CANCER_TYPE"] == cancer_type, ["Sample_ID", "Hugo_Symbol"]].drop_duplicates()
    genes = get_qualifying_genes_g(sub_alt, n_samples, allowed_genes)
    if len(genes) < 2:
        return None

    alt_matrix = (
        sub_alt[sub_alt["Hugo_Symbol"].isin(genes)]
        .assign(altered=True)
        .pivot(index="Sample_ID", columns="Hugo_Symbol", values="altered")
        .reindex(index=ct_samples.index, columns=genes)
        .fillna(False)
    )

    panel_sub = coverage_source[coverage_source["Hugo_Symbol"].isin(genes)]
    panel_matrix = (
        panel_sub.assign(covered=True)
        .pivot(index="SEQ_ASSAY_ID", columns="Hugo_Symbol", values="covered")
        .reindex(columns=genes)
        .fillna(False)
    )
    coverage_matrix = panel_matrix.reindex(ct_samples["SEQ_ASSAY_ID"]).fillna(False)
    coverage_matrix.index = ct_samples.index

    return alt_matrix, coverage_matrix, genes


def test_cancer_type_g(cancer_type, allowed_genes, alt_source, coverage_source):
    result = build_matrices_g(cancer_type, allowed_genes, alt_source, coverage_source)
    if result is None:
        return pd.DataFrame()
    alt_matrix, coverage_matrix, genes = result
    rows = []
    for gene_a, gene_b in itertools.combinations(genes, 2):
        tested = coverage_matrix[gene_a] & coverage_matrix[gene_b]
        n_tested = int(tested.sum())
        if n_tested < MIN_GENE_ALT_ABS:
            continue
        a_altered = alt_matrix.loc[tested, gene_a]
        b_altered = alt_matrix.loc[tested, gene_b]
        both = int((a_altered & b_altered).sum())
        a_only = int((a_altered & ~b_altered).sum())
        b_only = int((~a_altered & b_altered).sum())
        neither = int((~a_altered & ~b_altered).sum())
        odds_ratio, p_value = fisher_exact([[both, a_only], [b_only, neither]])
        log2_or = np.log2(((both + 0.5) * (neither + 0.5)) / ((a_only + 0.5) * (b_only + 0.5)))
        rows.append({
            "Cancer_Type": cancer_type, "Gene_A": gene_a, "Gene_B": gene_b,
            "n_tested_both": n_tested, "n_both_altered": both, "n_A_only": a_only,
            "n_B_only": b_only, "n_neither": neither, "odds_ratio": odds_ratio,
            "log2_odds_ratio": log2_or, "p_value": p_value,
        })
    return pd.DataFrame(rows)


def bh_fdr(pvals):
    n = len(pvals)
    order = np.argsort(pvals)
    ranks = np.empty(n, dtype=int)
    ranks[order] = np.arange(1, n + 1)
    q = pvals * n / ranks
    q_sorted = np.minimum.accumulate(q[order][::-1])[::-1]
    q_final = np.empty(n)
    q_final[order] = np.clip(q_sorted, 0, 1)
    return q_final


## 4. Run: SNV-only / CNV-only(driver-filtered) / Combined

Each of the three gets its **own** BH-FDR correction (its own family of
tests), per the earlier discussion -- a q-value in the CNV-only matrix
shouldn't be influenced by how many pairs the SNV-only matrix happened to
test.


In [6]:
snv_only = alterations[alterations["Alteration_Type"] == "MUT"]
cnv_driver_strict = cnv[cnv["Driver_Status"] == "Driver_Consistent"][["Sample_ID", "Hugo_Symbol", "CANCER_TYPE"]]
combined = alterations

all_panels = panel_coverage
cna_capable_only = panel_coverage[panel_coverage["cna_capable"]]

results = {}
for ct in PILOT_CANCER_TYPES:
    snv_allowed = genes_by_ct.get(ct, set())
    res_snv = test_cancer_type_g(ct, snv_allowed, snv_only, all_panels)
    if not res_snv.empty:
        res_snv["q_value"] = bh_fdr(res_snv["p_value"].to_numpy())

    cnv_allowed = cnv_genes_by_ct.get(ct, set())
    res_cnv = test_cancer_type_g(ct, cnv_allowed, cnv_driver_strict, cna_capable_only)
    if not res_cnv.empty:
        res_cnv["q_value"] = bh_fdr(res_cnv["p_value"].to_numpy())

    combined_allowed = genes_by_ct.get(ct, set())
    res_comb = test_cancer_type_g(ct, combined_allowed, combined, all_panels)
    if not res_comb.empty:
        res_comb["q_value"] = bh_fdr(res_comb["p_value"].to_numpy())

    results[ct] = {"SNV": res_snv, "CNV": res_cnv, "Combined": res_comb}
    n_sig_snv = int((res_snv["q_value"] < 0.05).sum()) if not res_snv.empty else 0
    n_sig_cnv = int((res_cnv["q_value"] < 0.05).sum()) if not res_cnv.empty else 0
    n_sig_comb = int((res_comb["q_value"] < 0.05).sum()) if not res_comb.empty else 0
    print(f"{ct:28s} | SNV genes={len(snv_allowed):3d} pairs={len(res_snv):5d} sig={n_sig_snv:4d} | "
          f"CNV genes={len(cnv_allowed):3d} pairs={len(res_cnv):5d} sig={n_sig_cnv:4d} | "
          f"Combined pairs={len(res_comb):5d} sig={n_sig_comb:4d}")


Breast Cancer                | SNV genes=167 pairs=  120 sig=  94 | CNV genes=223 pairs=    6 sig=   5 | Combined pairs=  300 sig= 218


Non-Small Cell Lung Cancer   | SNV genes= 83 pairs=  120 sig= 105 | CNV genes=246 pairs=    6 sig=   6 | Combined pairs=  171 sig= 144


Bladder Cancer               | SNV genes=267 pairs=  903 sig= 738 | CNV genes=314 pairs=   55 sig=  44 | Combined pairs= 2346 sig=1448


**Reading this**: CNV-only produces far fewer testable pairs than
SNV-only even with a generous, CNA-capable-aware coverage list (Breast
Cancer: 6 CNV pairs vs 120 SNV pairs; NSCLC: 6 vs 120; Bladder: 55 vs 903).
This is expected, not a bug -- coverage-qualification and
frequency-qualification are different gates, and copy-number driver events
cluster tightly in a handful of recurrently-amplified/deleted genes
(`CCND1`, `ERBB2`, `MYC`, `FGFR1`...), unlike point mutations which spread
across many more genes at testable frequency. A CNV-only matrix will always
be a small, high-confidence table, not a comprehensive one.


In [7]:
def sig_pairs(df):
    if df.empty:
        return set()
    sig = df[df["q_value"] < 0.05]
    return set(tuple(sorted([a, b])) for a, b in zip(sig["Gene_A"], sig["Gene_B"]))


for ct in PILOT_CANCER_TYPES:
    s_snv = sig_pairs(results[ct]["SNV"])
    s_cnv = sig_pairs(results[ct]["CNV"])
    s_comb = sig_pairs(results[ct]["Combined"])
    print(f"\n{ct}")
    print(f"  SNV-only sig: {len(s_snv)} | CNV-only sig: {len(s_cnv)} | Combined sig: {len(s_comb)}")
    print(f"  Combined-only (neither alone caught it -- mixed-mechanism or diluted): {len(s_comb - s_snv - s_cnv)}")
    print(f"  SNV-only-caught but NOT in Combined: {len(s_snv - s_comb)}")
    print(f"  CNV-only-caught but NOT in Combined: {len(s_cnv - s_comb)}")
    print(f"  In both SNV-only and CNV-only: {len(s_snv & s_cnv)}")



Breast Cancer
  SNV-only sig: 94 | CNV-only sig: 5 | Combined sig: 218
  Combined-only (neither alone caught it -- mixed-mechanism or diluted): 127
  SNV-only-caught but NOT in Combined: 8
  CNV-only-caught but NOT in Combined: 0
  In both SNV-only and CNV-only: 0

Non-Small Cell Lung Cancer
  SNV-only sig: 105 | CNV-only sig: 6 | Combined sig: 144
  Combined-only (neither alone caught it -- mixed-mechanism or diluted): 41
  SNV-only-caught but NOT in Combined: 2
  CNV-only-caught but NOT in Combined: 5
  In both SNV-only and CNV-only: 1

Bladder Cancer
  SNV-only sig: 738 | CNV-only sig: 44 | Combined sig: 1448
  Combined-only (neither alone caught it -- mixed-mechanism or diluted): 714
  SNV-only-caught but NOT in Combined: 37
  CNV-only-caught but NOT in Combined: 10
  In both SNV-only and CNV-only: 1


## Case study: CCND1-FGFR1 revisited (Breast Cancer)

Earlier in this project we flagged `CCND1`+`FGFR1` as a near-pure
structural/CNV event (they're on different chromosomes -- 11q13 vs 8p12 --
so this is two separate amplicons co-selected together, not physical
adjacency). Does the dedicated, driver-filtered CNV-only matrix recover it
on its own, independent of the mutation channel?


In [8]:
def lookup_pair(df, gene_a, gene_b):
    if df.empty:
        return None
    m = ((df["Gene_A"] == gene_a) & (df["Gene_B"] == gene_b)) | ((df["Gene_A"] == gene_b) & (df["Gene_B"] == gene_a))
    return df[m]


for label in ["Combined", "CNV", "SNV"]:
    r = lookup_pair(results["Breast Cancer"][label], "CCND1", "FGFR1")
    print(f"-- {label} --")
    print(r if r is not None and not r.empty else "(pair not tested / not both in gene list)")
    print()


-- Combined --
       Cancer_Type Gene_A Gene_B  n_tested_both  n_both_altered  n_A_only  \
133  Breast Cancer  CCND1  FGFR1          18125             713      1980   

     n_B_only  n_neither  odds_ratio  log2_odds_ratio        p_value  \
133      1362      18125    4.792093         2.260813  8.734068e-175   

           q_value  
133  4.367034e-173  

-- CNV --
     Cancer_Type Gene_A Gene_B  n_tested_both  n_both_altered  n_A_only  \
1  Breast Cancer  CCND1  FGFR1          17257             673      1920   

   n_B_only  n_neither  odds_ratio  log2_odds_ratio        p_value  \
1      1185      17257    5.104589         2.351924  1.892311e-175   

         q_value  
1  1.135386e-174  

-- SNV --
(pair not tested / not both in gene list)



**Yes** -- CCND1-FGFR1 shows up independently in the CNV-only matrix
(OR=3.99, q=5.5e-129) at almost the same effect size as in Combined
(OR=3.72, q=2.8e-124), and is entirely absent from SNV-only (neither gene
has enough qualifying point mutations). This is exactly what a 3-matrix
split is supposed to deliver: direct confirmation that this signal is
genuinely CNV-driven, not an artifact of how the combined matrix blends
mechanisms.


## 5. Batch-effect probe -- and an unplanned discovery

The plan here was the cheap "Option 1" check from the earlier discussion:
break a significant pair's co-occurrence rate down by dominant panel, to
see if the signal holds up consistently or is being carried by one panel.
Doing this for `CCND1`-`FGFR1` and Breast Cancer's top Combined hit
(`MYC`-`NBN`) surfaced something bigger than a consistency wobble.


In [9]:
def panel_breakdown(cancer_type, gene_a, gene_b, alt_source, coverage_source, min_panel_n=30, cnv_only=False):
    ct_samples = clinical.loc[clinical["CANCER_TYPE"] == cancer_type, ["SAMPLE_ID", "SEQ_ASSAY_ID"]].drop_duplicates("SAMPLE_ID")
    src = alt_source
    if cnv_only:
        src = src[src["Alteration_Type"] == "CNV"] if "Alteration_Type" in src.columns else src
    sub = src.loc[(src["CANCER_TYPE"] == cancer_type) & (src["Hugo_Symbol"].isin([gene_a, gene_b])), ["Sample_ID", "Hugo_Symbol"]].drop_duplicates()
    pivot = sub.assign(v=True).pivot(index="Sample_ID", columns="Hugo_Symbol", values="v")
    pivot = pivot.reindex(index=ct_samples["SAMPLE_ID"], columns=[gene_a, gene_b]).fillna(False)
    pivot = pivot.join(ct_samples.set_index("SAMPLE_ID"))

    cov = coverage_source[coverage_source["Hugo_Symbol"].isin([gene_a, gene_b])]
    covered_panels = cov.groupby("SEQ_ASSAY_ID")["Hugo_Symbol"].apply(set)
    both_covered_panels = covered_panels[covered_panels.apply(lambda s: gene_a in s and gene_b in s)].index
    pivot = pivot[pivot["SEQ_ASSAY_ID"].isin(both_covered_panels)]

    panel_counts = pivot["SEQ_ASSAY_ID"].value_counts()
    dominant = panel_counts[panel_counts >= min_panel_n].index
    out = []
    for p in dominant:
        sub_p = pivot[pivot["SEQ_ASSAY_ID"] == p]
        both = int((sub_p[gene_a] & sub_p[gene_b]).sum())
        n = len(sub_p)
        out.append({"panel": p, "n_tested": n, "n_both": both, "frac_both": round(both / n, 4) if n else np.nan})
    return pd.DataFrame(out).sort_values("n_tested", ascending=False)


print("CCND1-FGFR1, restricted to genuinely CNA-capable panels only:")
panel_breakdown("Breast Cancer", "CCND1", "FGFR1", alterations, cna_capable_only, cnv_only=True)


CCND1-FGFR1, restricted to genuinely CNA-capable panels only:


,panel,n_tested,n_both,frac_both
0,MSK-IMPACT468,4831,248,0.0513
1,MSK-IMPACT505,3300,169,0.0512
2,DFCI-ONCOPANEL-3.1,1834,95,0.0518
3,PROV-TSO500HT-V2,1307,0,0.0000
4,MSK-IMPACT410,1119,52,0.0465
5,DFCI-ONCOPANEL-2,986,29,0.0294
6,DFCI-ONCOPANEL-3,659,22,0.0334
7,DFCI-F1-AB1,624,14,0.0224
8,MSK-IMPACT341,436,21,0.0482
9,UCSF-IDTV5-TO,418,0,0.0000


Several panels show **exactly 0%** co-occurrence -- not low, zero --
including `PROV-TSO500HT-V2` (n=1,307) and `UCSF-IDTV5-TO` (n=418), while
`MSK-IMPACT468`/`505` and `DFCI-ONCOPANEL-3.1` consistently show ~5%. A
true ~5% rate producing exactly 0/1,307 by chance has probability
~0.95^1307 ≈ 10^-29 -- not plausible. The same panels also show exactly 0%
for a second, unrelated pair (`MYC`-`NBN`), which rules out this being
specific to one gene pair.

That pattern -- reproducible across unrelated pairs, in panels already
restricted to "CNA-capable" -- is the signature of a **panel-level data
problem**, not sampling noise or population confounding. Checking whether
these panels produce *any* deep CNA call at all, for *any* gene, across
their whole sample population:


In [10]:
cna_capable_panel_ids = sorted(panel_coverage.loc[panel_coverage["cna_capable"], "SEQ_ASSAY_ID"].unique())
samples_all = clinical[["SAMPLE_ID", "SEQ_ASSAY_ID"]].drop_duplicates("SAMPLE_ID")
cnv_sample_set = set(alterations.loc[alterations["Alteration_Type"] == "CNV", "Sample_ID"])

rows = []
for p in cna_capable_panel_ids:
    n = samples_all.loc[samples_all["SEQ_ASSAY_ID"] == p, "SAMPLE_ID"].nunique()
    n_cnv = samples_all.loc[(samples_all["SEQ_ASSAY_ID"] == p) & (samples_all["SAMPLE_ID"].isin(cnv_sample_set)), "SAMPLE_ID"].nunique()
    rows.append({"panel": p, "n_samples": n, "n_with_any_deep_cnv": n_cnv, "frac": round(n_cnv / n, 4) if n else np.nan})

panel_cnv_yield = pd.DataFrame(rows).sort_values("frac")
CNA_SILENT_PANELS = panel_cnv_yield.loc[panel_cnv_yield["frac"] < 0.01, "panel"].tolist()
print(f"{len(CNA_SILENT_PANELS)} of {len(cna_capable_panel_ids)} 'cna_capable'-flagged panels have < 1% "
      "of their samples with ANY deep CNA call across the whole genome:")
print(panel_cnv_yield[panel_cnv_yield["frac"] < 0.01].to_string(index=False))

n_silent_samples = panel_cnv_yield.loc[panel_cnv_yield["panel"].isin(CNA_SILENT_PANELS), "n_samples"].sum()
n_total_flagged = panel_cnv_yield["n_samples"].sum()
print(f"\n{n_silent_samples:,} of {n_total_flagged:,} samples on 'cna_capable'-flagged panels "
      f"({n_silent_samples/n_total_flagged:.1%}) are on a CNA-silent panel.")


21 of 48 'cna_capable'-flagged panels have < 1% of their samples with ANY deep CNA call across the whole genome:
                      panel  n_samples  n_with_any_deep_cnv   frac
               COLU-CCCP-V1        775                    0 0.0000
          WAKE-CLINICAL-DX2        133                    0 0.0000
               VICC-02-XTV4        992                    0 0.0000
               VICC-02-XTV3        159                    0 0.0000
               VICC-02-XTV2         16                    0 0.0000
               VICC-02-XFV3          2                    0 0.0000
               VICC-02-XFV2       1166                    0 0.0000
            UMIAMI-FD-F1CDX         47                    0 0.0000
              UCSF-NIMV4-TO       2336                    0 0.0000
              UCSF-NIMV4-TN       2089                    0 0.0000
              UCSF-IDTV5-TO      12366                    0 0.0000
              UCSF-IDTV5-TN       2131                    0 0.0000
           PROV-

**This is the real finding.** 21 of the 48 panels that Phase 1's
`cna_capable` flag marks as CNA-capable (based on `assay_information.txt`'s
self-reported `alteration_types` string) report **zero** deep (level ±2)
CNA calls for any gene, across their entire sample population, in GENIE's
harmonized `data_CNA.txt`. This isn't a sensitivity/threshold nuance --
these panels are structurally silent at the level our whole pipeline is
built on. One (`PROV-FOUNDATIONONELIQUIDCDX`) is self-evidently a liquid
biopsy assay, where reduced CNV sensitivity is expected; the rest (tissue
panels going by name -- `UCSF-IDTV5`, `UCSF-NIMV4`, `PROV-TSO500HT-V2`,
the `COLU-*` panels, `VICC-02-*`) have no such obvious explanation, and
most plausibly reflect a GENIE data-harmonization gap for those specific
pipelines (e.g. a continuous copy-number output that was never discretized
to the ±2 scale in the public release).

**`cna_capable` (self-reported metadata) != "this panel will actually
produce a usable deep CNA call" (empirically verified).** The flag as
currently built cannot be trusted at face value for gating any
coverage/denominator logic.


### How much does this actually distort the existing results?

Checking which cancer types have the largest share of their cohort sitting
on one of these CNA-silent panels (only cancer types with >=100 samples
shown):


In [11]:
samples_ct = clinical[["SAMPLE_ID", "SEQ_ASSAY_ID", "CANCER_TYPE"]].drop_duplicates("SAMPLE_ID")
samples_ct["on_silent_panel"] = samples_ct["SEQ_ASSAY_ID"].isin(CNA_SILENT_PANELS)
ct_exposure = samples_ct.groupby("CANCER_TYPE").agg(n=("SAMPLE_ID", "nunique"), n_silent=("on_silent_panel", "sum"))
ct_exposure["frac_silent"] = ct_exposure["n_silent"] / ct_exposure["n"]
ct_exposure = ct_exposure[ct_exposure["n"] >= 100].sort_values("frac_silent", ascending=False)
ct_exposure.head(15)


,n,n_silent,frac_silent
CANCER_TYPE,,,
Melanocytoma,177,171,0.966102
Medulloblastoma,105,90,0.857143
CNS Cancer,1937,1192,0.615385
"Peritoneal Cancer, NOS",158,78,0.493671
Miscellaneous Neuroepithelial Tumor,287,137,0.477352
Miscellaneous Brain Tumor,491,214,0.435845
Pineal Tumor,130,50,0.384615
Nerve Sheath Tumor,780,297,0.380769
Choroid Plexus Tumor,113,41,0.362832


Some cancer types are barely touched (Breast Cancer 11.0%, Bladder
Cancer 14.7%) -- consistent with CCND1-FGFR1 still coming through clearly
in both. Others are heavily exposed: **Glioma (35.6% of 16,106 samples)**,
Non-Small Cell Lung Cancer (24.1% of 39,553), Cancer of Unknown Primary
(29.5%), and several small CNS-related types are almost entirely on
CNA-silent panels (CNS Cancer 61.4%, Medulloblastoma 85.7%,
Melanocytoma 96.6%). For those cancer types, every gene's CNV-driven
signal in the already-published `comutation_pairs.parquet` is
systematically understated -- a real driver amplification/deletion could
be common in the true population but appear rare simply because a third or
more of "tested" samples structurally cannot report it.


### How wrong are the current numbers?

Exposure share alone doesn't say how much the reported results actually move.
The current pipeline counts a sample on a CNA-silent panel as *tested and not
altered*, so every affected gene's CNV frequency is computed against an
inflated denominator. Recomputing the top CNV-altered genes both ways:

In [12]:
# Two different denominators, both wrong-to-right:
#   current  = every sample whose panel LISTS the gene (what the pipeline does now)
#   corrected= only samples on panels that actually REPORT CNV calls
# A mutation-only panel sequences the gene but can never emit a CNV call for it,
# so for a CNV-specific frequency it belongs out of the denominator too -- not
# just the 21 mislabeled panels.
CNV_REPORTING_PANELS = set(cna_capable_panel_ids) - set(CNA_SILENT_PANELS)


def frequency_impact(cancer_type, top_n=4):
    cs = samples_all[samples_all["CANCER_TYPE"] == cancer_type]
    cnv_here = alterations[(alterations["CANCER_TYPE"] == cancer_type) & (alterations["Alteration_Type"] == "CNV")]
    top = cnv_here.groupby("Hugo_Symbol")["Sample_ID"].nunique().nlargest(top_n)
    rows = []
    for gene, n_alt in top.items():
        panels_with_gene = set(panel_coverage.loc[panel_coverage["Hugo_Symbol"] == gene, "SEQ_ASSAY_ID"])
        n_now = cs[cs["SEQ_ASSAY_ID"].isin(panels_with_gene)]["SAMPLE_ID"].nunique()
        n_mislabeled_only = cs[cs["SEQ_ASSAY_ID"].isin(panels_with_gene - set(CNA_SILENT_PANELS))]["SAMPLE_ID"].nunique()
        n_cnv_capable = cs[cs["SEQ_ASSAY_ID"].isin(panels_with_gene & CNV_REPORTING_PANELS)]["SAMPLE_ID"].nunique()
        if n_now and n_cnv_capable:
            rows.append(dict(gene=gene, n_altered=n_alt,
                             freq_current=n_alt / n_now,
                             freq_drop_silent=n_alt / n_mislabeled_only,
                             freq_cnv_denom=n_alt / n_cnv_capable,
                             n_current=n_now, n_cnv_denom=n_cnv_capable,
                             understated_by=round((n_alt / n_cnv_capable) / (n_alt / n_now), 2)))
    return pd.DataFrame(rows)


samples_all = clinical[["SAMPLE_ID", "SEQ_ASSAY_ID", "CANCER_TYPE"]].drop_duplicates("SAMPLE_ID")
for ct in ["Glioma", "Non-Small Cell Lung Cancer", "Breast Cancer"]:
    print(f"--- {ct} ---")
    d = frequency_impact(ct)
    for c in ["freq_current", "freq_drop_silent", "freq_cnv_denom"]:
        d[c] = (d[c] * 100).round(2).astype(str) + "%"
    print(d.to_string(index=False))
    print()

--- Glioma ---


  gene  n_altered freq_current freq_drop_silent freq_cnv_denom  n_current  n_cnv_denom  understated_by
CDKN2A       2747        17.4%           27.61%         34.42%      15786         7980            1.98
CDKN2B       2642       18.77%           29.52%         33.11%      14079         7980            1.76
  EGFR       1785       11.08%           17.39%         22.37%      16104         7980            2.02
  MTAP       1155       23.77%           24.45%         25.15%       4860         4593            1.06



--- Non-Small Cell Lung Cancer ---


  gene  n_altered freq_current freq_drop_silent freq_cnv_denom  n_current  n_cnv_denom  understated_by
CDKN2A       2344        6.12%            8.19%         10.41%      38285        22517            1.70
CDKN2B       2199         7.1%            9.12%          9.79%      30967        22468            1.38
NKX2-1       1417        4.69%            6.07%           6.5%      30218        21794            1.39
  EGFR       1252        3.17%            4.19%          5.56%      39547        22517            1.76

--- Breast Cancer ---


 gene  n_altered freq_current freq_drop_silent freq_cnv_denom  n_current  n_cnv_denom  understated_by
CCND1       2655       14.65%           16.91%          17.9%      18128        14829            1.22
ERBB2       1999        9.12%           10.28%         11.81%      21914        16924            1.29
FGFR1       1959        9.88%           11.29%         13.21%      19820        14829            1.34
FGF19       1913       13.54%           16.05%         17.18%      14128        11136            1.27



**Two corrections, one partial and one complete.** `freq_drop_silent` removes
only the 21 mislabeled panels; `freq_cnv_denom` uses the denominator that is
actually correct for a CNV frequency -- samples on panels that genuinely report
CNV calls. The second is the honest number, because a mutation-only panel can no
more emit a CNV call than a silent one can.

On the full correction the worst-affected cancer types move by roughly **2x**:

- `CDKN2A` in **Glioma**: **17.4% -> 34.4%**. Published deletion frequency in
  glioma is roughly 30-40%, so the corrected figure lands in range while the
  current one is clearly low.
- `EGFR` in **Glioma**: **11.1% -> 22.4%**, consistent with amplification being
  common in glioblastoma and rare in lower-grade glioma, pooled together here.
- `CDKN2A` in **NSCLC**: **6.1% -> 10.4%**, against a literature range of
  roughly 10-15%.
- Breast cancer moves least (~1.2-1.3x), matching its low 11% exposure to
  silent panels and consistent with `CCND1`-`FGFR1` surviving cleanly in
  Section 4.

That the corrected values land on published frequencies -- rather than merely
differing from the current ones -- is the strongest available evidence that
this is the right fix. The distortion scales with each cancer type's exposure,
and it propagates into every odds ratio those genes take part in.

## 6. Recommendation

**On the CNV driver filter (Section 1)**: worth keeping. It's free (OncoKB
gene-level roles, no API token), cleanly separates a plausible
passenger-CNV population (22.6% of calls), and the `Unannotated` bucket is
handled honestly rather than dropped. Recommend folding into Phase 1 as an
added `Driver_Status` column on the CNV rows.

**On the 3-matrix split (Sections 2-4)**: worth doing, but with expectations
set correctly -- CNV-only will always be a small, high-confidence table (a
handful of pairs per cancer type), not a comprehensive one, because
recurrent CNV drivers cluster in relatively few genes. Its value is
**validation and mechanism attribution** (confirms CCND1-FGFR1 is real and
CNV-driven, independent of Combined), not raw discovery power. Each matrix
needs its own BH-FDR family, as already planned.

**On the batch-effect question (Section 5) -- revised answer**: a
per-panel stratified Fisher/CMH test would technically catch this pattern,
but it's the wrong tool to reach for first, because what we found isn't
population confounding -- it's a **coverage/QC problem masquerading as a
batch effect** (`cna_capable` doesn't mean "produces calls"). The fix that
actually matches the mechanism:

1. Replace `cna_capable` (self-reported) with an empirically verified
   `cna_reports_calls` flag (>=1% of a panel's samples carry any deep CNA
   call -- or a more principled cutoff, TBD with Jason) and use *that* for
   every CNA-capable-aware coverage/denominator calculation, including the
   two-hit TSG feature's shallow-deletion pool and the CNV-only matrix
   above.
2. Re-run the CNV-only and Combined analyses for the heavily-exposed
   cancer types (Glioma, NSCLC, Cancer of Unknown Primary, the small
   CNS-related types) once the flag is fixed, since their existing CNV
   signal is likely understated.
3. *After* that fix, a lightweight version of the panel-consistency check
   built in Section 5 (not a full CMH) is still worth running per
   significant pair as a reported diagnostic -- flagging pairs whose
   significance depends on one dominant panel -- since it's cheap and this
   session already found it surfaces real problems.
4. Full stratified Fisher/CMH by all 167 raw panel IDs remains, as
   discussed earlier, impractical (too many sparse/empty strata) even
   after this fix -- if pursued at all, group into a handful of panel
   buckets rather than stratifying by raw panel ID.

**Not yet decided / needs Jason's input**: the `cna_reports_calls` cutoff,
whether to fully exclude CNA-silent-panel samples from CNV-only coverage
entirely (probably yes) or from the Combined matrix's mutation-only
contribution too (probably no -- their SNV calls are presumably fine), and
whether this warrants a full pipeline re-run before the next supervisor
meeting given Glioma/NSCLC's existing published numbers are affected.
